In [ ]:
import torch
from matmul_ale_ccp_log import Pi_update_ccp_log
from matmul_ale_ccp import (
    build_ccp_from_single_tree, build_species_helpers, 
    build_clade_species_mapping, build_ccp_helpers,
    get_root_clade_id, E_step
)

In [1120]:
delta, lambda_param, tau = 0.16103, 1e-10, 0.156391
iters = 100
species_tree_path = "/home/enzo/Documents/git/WP2/gpurec/test_mixed_200/sp.nwk"
gene_tree_path = "/home/enzo/Documents/git/WP2/gpurec/test_mixed_200/g.nwk"
dtype = torch.float64


In [1121]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🧮 Log-Space CCP Reconciliation")
print(f"Device: {device}")
print(f"Parameters: δ={delta}, τ={tau}, λ={lambda_param}")
print()

# Build CCP from gene tree
print("📊 Building CCP...")
ccp = build_ccp_from_single_tree(gene_tree_path)
print(f"   └─ {len(ccp.clades)} clades, {len(ccp.splits)} split groups")

# Build species helpers
print("🌳 Building species helpers...")
species_helpers = build_species_helpers(species_tree_path, device, dtype)
print(f"   └─ {species_helpers['S']} species nodes")

# Build clade-species mapping
print("🗺️  Building clade-species mapping...")
clade_species_map = build_clade_species_mapping(ccp, species_helpers, device, dtype)

# Build CCP helpers for GPU parallelization
print("⚡ Building CCP helpers...")
ccp_helpers = build_ccp_helpers(ccp, device, dtype)
total_splits = sum(len(splits) for splits in ccp.splits.values())
print(f"   └─ {len(ccp_helpers['split_parents'])} vectorized splits")

# Compute event probabilities
rates_sum = 1.0 + delta + tau + lambda_param
p_S = 1.0 / rates_sum
p_D = delta / rates_sum
p_T = tau / rates_sum
p_L = lambda_param / rates_sum

print(f"Event probabilities: S={p_S:.6f}, D={p_D:.6f}, T={p_T:.6f}, L={p_L:.6f}")

# Initialize matrices
S = species_helpers["S"]
C = len(ccp.clades)
E = torch.zeros(S, dtype=dtype, device=device)

# Initialize log_Pi in log space (start with very small probabilities)
log_Pi = torch.full((C, S), float('-inf'), dtype=dtype, device=device)

# Set leaf probabilities based on clade-species mapping
# If a leaf clade maps to a species, set log probability to 0 (probability = 1)
leaf_count = 0
for c in range(C):
    clade = ccp.id_to_clade[c]
    if clade.is_leaf():
        leaf_count += 1
        # Find which species this leaf belongs to
        mapped_species = torch.nonzero(clade_species_map[c] > 0, as_tuple=False).flatten()
        if len(mapped_species) > 0:
            # Distribute probability uniformly among mapped species
            log_prob = -torch.log(torch.tensor(len(mapped_species), dtype=dtype))
            log_Pi[c, mapped_species] = log_prob
            if leaf_count <= 3:  # Debug first few leaves
                print(f"   Leaf {c} ({clade}): mapped to species {mapped_species.tolist()}, log_prob={log_prob:.6f}")
        else:
            if leaf_count <= 3:
                print(f"   Leaf {c} ({clade}): NO MAPPING FOUND!")

print(f"   Initialized {leaf_count} leaf clades")
print(f"   Non-infinite log_Pi values after init: {torch.isfinite(log_Pi).sum()}")

print(f"Matrix dimensions: Pi = {C} × {S} = {C*S:,} elements")
print()

# === EXTINCTION PROBABILITY COMPUTATION ===
print("💀 Computing extinction probabilities...")
for iter_e in range(iters):
    E_next, E_s1, E_s2, Ebar = E_step(E, species_helpers["s_C1"], species_helpers["s_C2"], 
                                        species_helpers["Recipients_mat"], p_S, p_D, p_T, p_L)
    E = E_next
print(f"   └─ Converged after {iters} iterations")

# === LIKELIHOOD COMPUTATION ===
print("🧮 Computing likelihood matrix...")
for iter_pi in range(iters):
    log_Pi_new = Pi_update_ccp_log(log_Pi, ccp_helpers, species_helpers, clade_species_map, 
                                    E, Ebar, p_S, p_D, p_T)
    
    # Check convergence in log space
    if iter_pi > 0:
        diff = torch.abs(log_Pi_new - log_Pi).max()
        if diff < 1e-10:
            print(f"   └─ Converged after {iter_pi+1} iterations")
            break
    
    log_Pi = log_Pi_new

# === COMPUTE LOG-LIKELIHOOD ===
root_clade_id = get_root_clade_id(ccp)
root_clade = ccp.id_to_clade[root_clade_id]

print(f"🌲 Root clade: {root_clade}")

# Compute log-likelihood: log(sum(Pi[root, :]))
log_likelihood = torch.logsumexp(log_Pi[root_clade_id, :], dim=0)

print(f"📊 Log-likelihood: {log_likelihood:.6f}")

# Check for numerical issues
print(f"   Root log Pi values: min={log_Pi[root_clade_id, :].min():.6f}, max={log_Pi[root_clade_id, :].max():.6f}")
print(f"   Non-finite values in log_Pi: {torch.logical_not(torch.isfinite(log_Pi)).sum()}")
print(f"   NaN values in log_Pi: {torch.isnan(log_Pi).sum()}")
print(f"   -inf values in log_Pi: {torch.isinf(log_Pi).sum()}")

if torch.isnan(log_likelihood) or torch.isinf(log_likelihood):
    print("⚠️  WARNING: Numerical instability detected!")
    
    # Debug: check leaf initialization
    leaf_indices = [i for i in range(len(ccp.id_to_clade)) if ccp.id_to_clade[i].is_leaf()]
    print(f"   Leaf clades: {len(leaf_indices)}")
    for i in leaf_indices[:3]:  # Show first 3 leaves
        print(f"     Leaf {i}: {ccp.id_to_clade[i]} -> log_Pi = {log_Pi[i, :].max():.6f}")
    
else:
    print("✅ No numerical instability detected")

    import torch.nn.functional as F


🧮 Log-Space CCP Reconciliation
Device: cuda
Parameters: δ=0.16103, τ=0.156391, λ=1e-10

📊 Building CCP...
Debug: Found 4669 root splits, expected 4669 for 2336 leaves
   └─ 9339 clades, 7003 split groups
🌳 Building species helpers...
   └─ 199 species nodes
🗺️  Building clade-species mapping...
⚡ Building CCP helpers...
   └─ 11671 vectorized splits
Event probabilities: S=0.759059, D=0.122231, T=0.118710, L=0.000000
   Leaf 123 (Clade(['65_504'], size=1)): mapped to species [0], log_prob=-0.000000
   Leaf 129 (Clade(['166_3445'], size=1)): mapped to species [6], log_prob=-0.000000
   Leaf 133 (Clade(['114_1773'], size=1)): mapped to species [12], log_prob=-0.000000
   Initialized 2336 leaf clades
   Non-infinite log_Pi values after init: 2336
Matrix dimensions: Pi = 9339 × 199 = 1,858,461 elements

💀 Computing extinction probabilities...
   └─ Converged after 100 iterations
🧮 Computing likelihood matrix...
   └─ Converged after 49 iterations
Root clade ID: 0, Root clade: Clade(['100_12

In [ ]:
import torch.nn.functional as F
theta_real = torch.tensor([delta, tau, lambda_param], dtype=dtype, device=device, requires_grad=True)
optimizer = torch.optim.Adam([theta_real], lr=0.01)
theta = F.softplus(theta_real)
p_S = 1.0 / (1.0 + theta.sum())
p_D = theta[0] / (1.0 + theta.sum())
p_T = theta[1] / (1.0 + theta.sum())
p_L = theta[2] / (1.0 + theta.sum())
for i in range(1000):
    theta = F.softplus(theta_real)
    p_S = 1.0 / (1.0 + theta.sum())
    p_D = theta[0] / (1.0 + theta.sum())
    p_T = theta[1] / (1.0 + theta.sum())
    p_L = theta[2] / (1.0 + theta.sum())

    E_new, E_s1, E_s2, Ebar_new = E_step(
        E, species_helpers["s_C1"], species_helpers["s_C2"],
        species_helpers["Recipients_mat"], p_S, p_D, p_T, p_L
    )


    log_Pi_new = Pi_update_ccp_log(
        log_Pi, ccp_helpers, species_helpers, clade_species_map,
        E, Ebar, p_S, p_D, p_T
    )
    with torch.no_grad():
        E = E_new.clone().detach()
        Ebar = Ebar_new.clone().detach()
        log_Pi = log_Pi_new.clone().detach()
    # Optimize theta_real by gradient ascent
    loss = -torch.logsumexp(log_Pi_new[0], dim=0)

    # print(f'theta_real: {theta_real.detach().cpu().numpy()}')
    loss.backward()
    #print(f"Gradient before step: {theta_real.grad}")
    optimizer.step()
    #print(f"Final theta_real: {theta_real.detach().cpu().numpy()}")
    if i < 10 or i % 100 == 0:
        print(f"Iteration {i+1}: loss = {loss.item():.6f}, "
              f"p_S = {p_S:.6f}, p_D = {p_D:.6f}, p_T = {p_T:.6f}, p_L = {p_L:.6f}")


In [ ]:
for i in range(1000):
    theta = F.softplus(theta_real)
    p_S = 1.0 / (1.0 + theta.sum())
    p_D = theta[0] / (1.0 + theta.sum())
    p_T = theta[1] / (1.0 + theta.sum())
    p_L = theta[2] / (1.0 + theta.sum())

    E_new, E_s1, E_s2, Ebar_new = E_step(
        E, species_helpers["s_C1"], species_helpers["s_C2"],
        species_helpers["Recipients_mat"], p_S, p_D, p_T, p_L
    )


    log_Pi_new = Pi_update_ccp_log(
        log_Pi, ccp_helpers, species_helpers, clade_species_map,
        E, Ebar, p_S, p_D, p_T
    )
    with torch.no_grad():
        E = E_new.clone().detach()
        Ebar = Ebar_new.clone().detach()
        log_Pi = log_Pi_new.clone().detach()
    # Optimize theta_real by gradient ascent
    loss = -torch.logsumexp(log_Pi_new[0], dim=0)

    # print(f'theta_real: {theta_real.detach().cpu().numpy()}')
    loss.backward()
    #print(f"Gradient before step: {theta_real.grad}")
    optimizer.step()
    #print(f"Final theta_real: {theta_real.detach().cpu().numpy()}")
    if i < 10 or i % 100 == 0:
        print(f"Iteration {i+1}: loss = {loss.item():.6f}, "
              f"p_S = {p_S:.6f}, p_D = {p_D:.6f}, p_T = {p_T:.6f}, p_L = {p_L:.6f}")


In [ ]:
theta_real = torch.tensor([delta, tau, lambda_param], dtype=dtype, device=device, requires_grad=True)
theta_real

In [1111]:
theta_real = torch.tensor([delta, tau, lambda_param], dtype=dtype, device=device, requires_grad=True)
optimizer = torch.optim.LBFGS([theta_real], lr=0.01, max_iter=20, line_search_fn=None)

for i in range(10000):
    def closure():
        optimizer.zero_grad()
        theta = F.softplus(theta_real)
        Z = 1.0 + theta.sum()
        p_S = 1.0 / Z
        p_D = theta[0] / Z
        p_T = theta[1] / Z
        p_L = theta[2] / Z
        E_new, E_s1, E_s2, Ebar_new = E_step(
            E, species_helpers["s_C1"], species_helpers["s_C2"],
            species_helpers["Recipients_mat"], p_S, p_D, p_T, p_L
        )

        log_Pi_new = Pi_update_ccp_log(
            log_Pi, ccp_helpers, species_helpers, clade_species_map,
            E, Ebar, p_S, p_D, p_T
        )

        loss = -torch.logsumexp(log_Pi_new[0], dim=0)
        loss.backward()

        return loss

    loss = optimizer.step(closure)

    # After LBFGS step, manually update model state
    with torch.no_grad():
        theta = F.softplus(theta_real)
        Z = 1.0 + theta.sum()
        p_S = 1.0 / Z
        p_D = theta[0] / Z
        p_T = theta[1] / Z
        p_L = theta[2] / Z

        E_new, E_s1, E_s2, Ebar_new = E_step(
            E, species_helpers["s_C1"], species_helpers["s_C2"],
            species_helpers["Recipients_mat"], p_S, p_D, p_T, p_L
        )
        log_Pi_new = Pi_update_ccp_log(
            log_Pi, ccp_helpers, species_helpers, clade_species_map,
            E, Ebar, p_S, p_D, p_T
        )

        E = E_new.clone().detach()
        Ebar = Ebar_new.clone().detach()
        log_Pi = log_Pi_new.clone().detach()

    if i < 10 or i % 1 == 0:
        print(f"Iteration {i+1}: loss = {loss.item():.6f}, "
              f"p_S = {p_S:.6e}, p_D = {p_D:.6e}, p_T = {p_T:.6e}, p_L = {p_L:.6e}")

Iteration 1: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 2: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 3: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 4: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 5: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 6: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 7: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 8: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 9: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 10: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 11: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 12: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 13: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 14: loss = inf, p_S = nan, p_D = nan, p_T = nan, p_L = nan
Iteration 15: loss = inf, p_S = nan, p_D = 

KeyboardInterrupt: 

In [1105]:
import torch.nn.functional as F

theta_real = torch.tensor([delta, tau, lambda_param], dtype=dtype, device=device, requires_grad=True)
optimizer = torch.optim.Adam([theta_real], lr=0.001)

for i in range(10000):
    optimizer.zero_grad()

    theta = F.softplus(theta_real)
    Z = 1.0 + theta.sum()
    p_S = 1.0 / Z
    p_D = theta[0] / Z
    p_T = theta[1] / Z
    p_L = theta[2] / Z

    E_new, E_s1, E_s2, Ebar_new = E_step(
        E, species_helpers["s_C1"], species_helpers["s_C2"],
        species_helpers["Recipients_mat"], p_S, p_D, p_T, p_L
    )

    log_Pi_new = Pi_update_ccp_log(
        log_Pi, ccp_helpers, species_helpers, clade_species_map,
        E, Ebar, p_S, p_D, p_T
    )

    loss = -torch.logsumexp(log_Pi_new[0], dim=0)
    loss.backward()
    optimizer.step()

    # Detach updated state
    with torch.no_grad():
        E = E_new.clone()
        Ebar = Ebar_new.clone()
        log_Pi = log_Pi_new.clone()

    if i < 10 or i % 100 == 0:
        print(f"Iteration {i+1}: loss = {loss.item():.6f}, "
              f"p_S = {p_S:.6e}, p_D = {p_D:.6e}, p_T = {p_T:.6e}, p_L = {p_L:.6e}")


Iteration 1: loss = 6775.382405, p_S = 3.092921e-01, p_D = 2.302360e-01, p_T = 2.302360e-01, p_L = 2.302360e-01
Iteration 2: loss = 6775.466508, p_S = 3.094428e-01, p_D = 2.301857e-01, p_T = 2.301857e-01, p_L = 2.301857e-01
Iteration 3: loss = 6775.554006, p_S = 3.095916e-01, p_D = 2.301400e-01, p_T = 2.301343e-01, p_L = 2.301340e-01
Iteration 4: loss = 6775.131970, p_S = 3.097390e-01, p_D = 2.300969e-01, p_T = 2.300828e-01, p_L = 2.300812e-01
Iteration 5: loss = 6775.140724, p_S = 3.098880e-01, p_D = 2.300509e-01, p_T = 2.300316e-01, p_L = 2.300295e-01
Iteration 6: loss = 6775.437135, p_S = 3.100359e-01, p_D = 2.300069e-01, p_T = 2.299803e-01, p_L = 2.299768e-01
Iteration 7: loss = 6776.553700, p_S = 3.101793e-01, p_D = 2.299727e-01, p_T = 2.299273e-01, p_L = 2.299208e-01
Iteration 8: loss = 6777.988039, p_S = 3.102838e-01, p_D = 2.300207e-01, p_T = 2.298596e-01, p_L = 2.298359e-01
Iteration 9: loss = 6779.534088, p_S = 3.103576e-01, p_D = 2.301202e-01, p_T = 2.297941e-01, p_L = 2.297